# Kaggle Phase 3 — Static Training, Baselines, Regime Comparison

Third of 4 notebooks. Requires TWO attached datasets (Add Input):
1. `kagglephase1-output` — features/windows/masks from `kagglephase1`
2. `kagglephase2-defs` — `model_defs.py` from `kagglephase2`

**What it does**
1. Trains the residual GRU for horizons 1–3 (15s/30s/45s ahead) on the burst-injected
   train split (static model).
2. Runs the non-learned baselines (persistence, SES with train-fit alpha).
3. Evaluates the static checkpoints on both test variants, including a spike-affected
   vs. normal window breakdown of the burst-injected test split.
4. Exports every metric (JSON + CSVs) and the trained checkpoints as one zipped
   download — **Phase 4 consumes this zip** for all streaming/drift-aware evaluation
   (DriftMonitor, OnlineAdapter, AdaptiveThreshold), which no longer lives here.


## Step 1: Input Checks + Imports

In [ ]:
import os, sys, gc, json, time
import numpy as np
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle')
inp = Path('/kaggle/input') if IN_KAGGLE else Path('.')

hits = sorted(inp.glob('**/features_injected.npy'), key=lambda p: len(p.parts))
if not hits:
    raise FileNotFoundError(
        "Phase-1 output not found under /kaggle/input -- attach the 'kagglephase1-output' "
        "dataset (produced by kagglephase1.ipynb) via 'Add Input', then re-run."
    )
P1 = hits[0].parent
defs_hits = sorted(inp.glob('**/model_defs.py'), key=lambda p: len(p.parts))
if not defs_hits:
    raise FileNotFoundError(
        "model_defs.py not found under /kaggle/input -- attach the 'kagglephase2-defs' "
        "dataset (produced by kagglephase2.ipynb) via 'Add Input', then re-run."
    )
sys.path.insert(0, str(defs_hits[0].parent))
from model_defs import (AdaptiveGRUModel, WindowDataset, collate_pad, make_loader, set_seed,
                        EarlyStopping, train_one_epoch, compute_metrics, predict_all, evaluate,
                        train_model, persistence_preds, ses_fit_alpha, ses_preds)
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Phase-1 input : {P1}")
print(f"model_defs    : {defs_hits[0]}")
print(f"device        : {device}")

manifest = json.load(open(P1 / 'manifest.json'))
fc = json.load(open(P1 / 'feature_cols.json'))
stats = json.load(open(P1 / 'normalization_stats.json'))
TARGET_NAMES = fc['target_names']
TARGET_IDX = fc['target_idx']
TARGET_MEAN = np.array([stats[c]['mean'] for c in fc['target_columns']])
TARGET_STD  = np.array([stats[c]['std']  for c in fc['target_columns']])
FEAT_INJ, FEAT_CLEAN = P1 / 'features_injected.npy', P1 / 'features_clean.npy'
windows = {k: np.load(P1 / f'windows_{k}.npy') for k in ('train', 'val', 'test')}
windows_test_clean = np.load(P1 / 'windows_test_clean.npy')
spike_mask = np.load(P1 / 'spike_mask.npy')
feat_inj_arr = np.load(FEAT_INJ, mmap_mode='r')
feat_clean_arr = np.load(FEAT_CLEAN, mmap_mode='r')
print({k: len(v) for k, v in windows.items()}, '| clean test:', len(windows_test_clean))


## Step 2: Config

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Phase3Config:
    horizons: List[int] = field(default_factory=lambda: [1, 2, 3])
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.2
    batch_size: int = 128
    eval_batch_size: int = 512
    lr: float = 1e-3
    epochs: int = 30
    patience: int = 7
    spike_lookback_tail: int = 60
    ckpt_dir: str = '/kaggle/working/phase3_checkpoints' if os.path.exists('/kaggle') else './phase3_checkpoints'

cfg = Phase3Config()
Path(cfg.ckpt_dir).mkdir(parents=True, exist_ok=True)
print(cfg)
print(f"Horizons: {cfg.horizons} (15s/30s/45s ahead). All streaming/drift-aware evaluation "
      f"(DriftMonitor, OnlineAdapter, AdaptiveThreshold) has moved to Phase 4, which consumes "
      f"this notebook's zipped output -- this notebook covers static training, baselines, and "
      f"the static regime comparison only.")

## Step 3: Train the Static Residual GRU (per horizon, burst-injected train split)

In [27]:
static_results = {}
for h in cfg.horizons:
    print(f"\n{'='*58}\nHORIZON {h}  ({h*15}s ahead)\n{'='*58}")
    set_seed(42 + h)
    tr_loader = make_loader(FEAT_INJ, windows['train'], h, TARGET_IDX,
                            batch_size=cfg.batch_size, shuffle=True)
    va_loader = make_loader(FEAT_INJ, windows['val'], h, TARGET_IDX,
                            batch_size=cfg.eval_batch_size)
    model = AdaptiveGRUModel(input_size=27, hidden_size=cfg.hidden_size,
                             num_layers=cfg.num_layers, dropout=cfg.dropout,
                             residual_indices=TARGET_IDX).to(device)
    print(f"  {model.n_params():,} params | train batches {len(tr_loader)} | val batches {len(va_loader)}")
    ckpt = str(Path(cfg.ckpt_dir) / f'gru_h{h}_static.pt')
    best = train_model(model, tr_loader, va_loader, device, ckpt,
                       epochs=cfg.epochs, lr=cfg.lr, patience=cfg.patience)
    model.load_state_dict(torch.load(ckpt, map_location=device)['model_state_dict'])
    model.eval()

    te_inj = make_loader(FEAT_INJ, windows['test'], h, TARGET_IDX, batch_size=cfg.eval_batch_size)
    te_cln = make_loader(FEAT_CLEAN, windows_test_clean, h, TARGET_IDX, batch_size=cfg.eval_batch_size)
    m_inj = evaluate(model, te_inj, device, TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    m_cln = evaluate(model, te_cln, device, TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    static_results[h] = {'best_val_loss': best, 'test_injected': m_inj, 'test_clean': m_cln}
    print(f"  clean-test MAPE mean={m_cln['mape_mean']:.2f}% | injected-test MAPE mean={m_inj['mape_mean']:.2f}%")
    del model, tr_loader, va_loader, te_inj, te_cln
    torch.cuda.empty_cache(); gc.collect()



HORIZON 1  (15s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.006615 | val=0.000512 | patience=0/7 | 112s
  epoch   5 | train=0.006134 | val=0.000345 | patience=0/7 | 556s
  epoch  10 | train=0.005833 | val=0.000482 | patience=4/7 | 1108s
  early stop at epoch 13
  done: best_val=0.000329 | 1439s
  clean-test MAPE mean=0.08% | injected-test MAPE mean=0.13%

HORIZON 2  (30s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.010480 | val=0.001050 | patience=0/7 | 111s
  epoch   5 | train=0.009259 | val=0.000803 | patience=1/7 | 557s
  epoch  10 | train=0.008183 | val=0.000671 | patience=3/7 | 1110s
  early stop at epoch 14
  done: best_val=0.000631 | 1552s
  clean-test MAPE mean=0.21% | injected-test MAPE mean=0.21%

HORIZON 3  (45s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.015125 | val=0.001447 | patience=0/7 | 112s
  epoch   5 | train=0.012654 | val=0.001359 | patience=1/7 

## Step 4: Non-Learned Baselines (persistence, SES) on Both Test Variants

In [29]:
def spike_window_mask(wins, horizon):
    tgt_rows = wins[:, 0] - 1 + horizon
    m = spike_mask[tgt_rows].copy()
    tail = cfg.spike_lookback_tail
    for i, (e, L) in enumerate(wins):
        if not m[i]:
            m[i] = spike_mask[max(e - tail, e - L):e].any()
    return m

baseline_results = {}
for h in cfg.horizons:
    row = {}
    for variant, feat_arr, wins in (('clean', feat_clean_arr, windows_test_clean),
                                    ('injected', feat_inj_arr, windows['test'])):
        tgt = np.asarray(feat_arr[wins[:, 0] - 1 + h])[:, TARGET_IDX].astype(np.float64)
        p = persistence_preds(feat_arr, wins, TARGET_IDX)
        alphas = ses_fit_alpha(feat_arr, windows['train'], h, TARGET_IDX, max_n=8000, seed=h)
        s = ses_preds(feat_arr, wins, TARGET_IDX, alphas)
        row[variant] = {
            'persistence': compute_metrics(p, tgt, TARGET_STD, TARGET_MEAN, TARGET_NAMES),
            'ses': compute_metrics(s, tgt, TARGET_STD, TARGET_MEAN, TARGET_NAMES),
            'ses_alphas': alphas,
        }
    baseline_results[h] = row
    print(f"h={h}: clean  P={row['clean']['persistence']['mape_mean']:.2f}%  "
          f"SES={row['clean']['ses']['mape_mean']:.2f}% (alpha={row['clean']['ses_alphas']}) | "
          f"injected  P={row['injected']['persistence']['mape_mean']:.2f}%  "
          f"SES={row['injected']['ses']['mape_mean']:.2f}%")


h=1: clean  P=0.05%  SES=0.06% (alpha=[np.float64(1.0), np.float64(0.9), np.float64(0.85), np.float64(0.85)]) | injected  P=0.13%  SES=0.15%
h=2: clean  P=0.11%  SES=0.12% (alpha=[np.float64(1.0), np.float64(0.85), np.float64(0.85), np.float64(0.85)]) | injected  P=0.26%  SES=0.28%
h=3: clean  P=0.16%  SES=0.17% (alpha=[np.float64(1.0), np.float64(0.75), np.float64(0.75), np.float64(0.85)]) | injected  P=0.40%  SES=0.41%
h=4: clean  P=0.21%  SES=0.21% (alpha=[np.float64(1.0), np.float64(0.6), np.float64(1.0), np.float64(1.0)]) | injected  P=0.52%  SES=0.52%
h=5: clean  P=0.24%  SES=0.24% (alpha=[np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]) | injected  P=0.62%  SES=0.62%


## Step 5: Static GRU Evaluation by Regime (spike vs. normal windows)

Batch (non-streaming) evaluation of the trained static checkpoints on the burst-injected
test split, separated into spike-affected vs. normal windows via `spike_window_mask`.
The streaming drift-aware evaluation — chronological chunks, DriftMonitor-triggered
OnlineAdapter fine-tunes, AdaptiveThreshold bands, static-vs-adaptive comparison — has
moved entirely to Phase 4, which loads the checkpoints exported at the end of this notebook.


In [ ]:
static_regime = {}
for h in cfg.horizons:
    wins = windows['test']
    loader = make_loader(FEAT_INJ, wins, h, TARGET_IDX, batch_size=cfg.eval_batch_size)
    ckpt = torch.load(str(Path(cfg.ckpt_dir) / f'gru_h{h}_static.pt'), map_location=device)
    model = AdaptiveGRUModel(input_size=27, hidden_size=cfg.hidden_size,
                             num_layers=cfg.num_layers, dropout=cfg.dropout,
                             residual_indices=TARGET_IDX).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    p, t = predict_all(model, loader, device)
    sm = spike_window_mask(wins, h)
    static_regime[h] = {
        'overall': compute_metrics(p, t, TARGET_STD, TARGET_MEAN, TARGET_NAMES),
        'spike':   compute_metrics(p[sm], t[sm], TARGET_STD, TARGET_MEAN, TARGET_NAMES),
        'normal':  compute_metrics(p[~sm], t[~sm], TARGET_STD, TARGET_MEAN, TARGET_NAMES),
        'n_spike_windows': int(sm.sum()), 'n_windows': len(wins),
    }
    r = static_regime[h]
    print(f"h={h}: overall MAPE={r['overall']['mape_mean']:.2f}% | "
          f"spike={r['spike']['mape_mean']:.2f}% ({r['n_spike_windows']:,} windows) | "
          f"normal={r['normal']['mape_mean']:.2f}%")
    del model, loader
    torch.cuda.empty_cache()
    gc.collect()

## Step 6: Final Comparison Tables + Metrics Export (zipped download)

Table A: clean test. Table B: burst-injected test, spike vs. normal windows (static
model only — the adaptive comparison is Phase 4's job). Then every metric is written
to `all_metrics.json` + two flat CSVs, bundled with the trained checkpoints into
`kagglephase3_output.zip` for download from the Output tab.


In [ ]:
import csv, zipfile

print("="*74)
print("TABLE A -- CLEAN (untouched) test data, MAPE % per target")
print("="*74)
for h in cfg.horizons:
    print(f"\nhorizon {h} ({h*15}s ahead)")
    print(f"  {'target':<18} {'Persist':>9} {'SES':>9} {'GRU-static':>11}")
    b = baseline_results[h]['clean']
    g = static_results[h]['test_clean']
    for n in TARGET_NAMES:
        print(f"  {n:<18} {b['persistence'][n]['mape']:>8.2f}% {b['ses'][n]['mape']:>8.2f}% "
              f"{g[n]['mape']:>10.2f}%")

print()
print("="*74)
print("TABLE B -- BURST-INJECTED test data (batch static eval), MAPE % per target")
print("="*74)
for h in cfg.horizons:
    b = baseline_results[h]['injected']
    r = static_regime[h]
    for regime in ('spike', 'normal'):
        n_reg = r['n_spike_windows'] if regime == 'spike' else r['n_windows'] - r['n_spike_windows']
        print(f"\nhorizon {h}, {regime.upper()} windows ({n_reg:,}):")
        print(f"  {'target':<18} {'Persist*':>9} {'SES*':>9} {'GRU-static':>11}")
        for n in TARGET_NAMES:
            print(f"  {n:<18} {b['persistence'][n]['mape']:>8.2f}% {b['ses'][n]['mape']:>8.2f}% "
                  f"{r[regime][n]['mape']:>10.2f}%")
print("\n(* baseline columns are whole-injected-test figures; the GRU column is per-regime.)")
print("The adaptive (streaming, drift-aware) comparison lives in Phase 4, which consumes the")
print("zip produced below.")

out_dir = Path('/kaggle/working') if os.path.exists('/kaggle') else Path('.')

ckpt_meta = {}
for p in sorted(Path(cfg.ckpt_dir).glob('*.pt')):
    ck = torch.load(str(p), map_location='cpu')
    ckpt_meta[p.name] = {'size_mb': round(p.stat().st_size / 1e6, 2),
                         'epoch': ck.get('epoch'), 'val_loss': ck.get('val_loss')}
print(f"\nCheckpoints on disk: {list(ckpt_meta)}")

all_metrics = {
    'config': {k: getattr(cfg, k) for k in vars(cfg)},
    'static_results': static_results,
    'baseline_results': baseline_results,
    'static_regime': static_regime,
    'checkpoints': ckpt_meta,
}
json_path = out_dir / 'all_metrics.json'
with open(json_path, 'w') as f:
    json.dump(all_metrics, f, indent=1, default=float)

summary_path = out_dir / 'all_metrics_summary.csv'
with open(summary_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['horizon', 'best_val_loss', 'test_clean_mape_mean', 'test_injected_mape_mean',
                'persist_clean_mape_mean', 'ses_clean_mape_mean',
                'persist_injected_mape_mean', 'ses_injected_mape_mean',
                'static_spike_mape_mean', 'static_normal_mape_mean',
                'n_spike_windows', 'n_windows'])
    for h in cfg.horizons:
        s, b, r = static_results[h], baseline_results[h], static_regime[h]
        w.writerow([h, s['best_val_loss'], s['test_clean']['mape_mean'], s['test_injected']['mape_mean'],
                    b['clean']['persistence']['mape_mean'], b['clean']['ses']['mape_mean'],
                    b['injected']['persistence']['mape_mean'], b['injected']['ses']['mape_mean'],
                    r['spike']['mape_mean'], r['normal']['mape_mean'],
                    r['n_spike_windows'], r['n_windows']])

detail_path = out_dir / 'all_metrics_per_target.csv'
with open(detail_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['horizon', 'target', 'variant', 'method', 'mae', 'rmse', 'mape'])
    for h in cfg.horizons:
        for variant_key, variant in (('test_clean', 'clean'), ('test_injected', 'injected')):
            m = static_results[h][variant_key]
            for tname in TARGET_NAMES:
                tm = m[tname]
                w.writerow([h, tname, variant, 'GRU-static', tm['mae'], tm['rmse'], tm['mape']])
        for variant in ('clean', 'injected'):
            for method in ('persistence', 'ses'):
                mm = baseline_results[h][variant][method]
                for tname in TARGET_NAMES:
                    tm = mm[tname]
                    w.writerow([h, tname, variant, method, tm['mae'], tm['rmse'], tm['mape']])
        for regime in ('spike', 'normal'):
            m = static_regime[h][regime]
            for tname in TARGET_NAMES:
                tm = m[tname]
                w.writerow([h, tname, f'injected_{regime}', 'GRU-static', tm['mae'], tm['rmse'], tm['mape']])

zip_path = out_dir / 'kagglephase3_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in (json_path, summary_path, detail_path):
        z.write(p, p.name)
    for p in Path(cfg.ckpt_dir).glob('*.pt'):
        z.write(p, f'checkpoints/{p.name}')

print(f"\nSaved: {json_path.name}, {summary_path.name}, {detail_path.name}")
print(f"Archive: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
print()
print("NEXT: Output tab -> download kagglephase3_output.zip -> unzip -> upload the files as a")
print("Kaggle Dataset named 'kagglephase3-output' -> attach it to Phase 4 via Add Input")
print("(alongside 'kagglephase1-output' and 'kagglephase2-defs').")

## How the Model Saves (checkpoint mechanism)

- **When**: during training (`train_model` in `model_defs.py`), after every epoch the
  validation loss is compared to the best seen so far. A checkpoint is written **only when
  a new best is reached** — so `gru_h{h}_static.pt` always holds the single
  best-validation version of that horizon's model, *not* the final epoch's weights
  (early stopping typically runs several epochs past the best one before halting).
- **Where**: one file per horizon under `/kaggle/working/phase3_checkpoints/`
  (`gru_h1_static.pt`, `gru_h2_static.pt`, `gru_h3_static.pt`).
- **What's inside**: `{'model_state_dict': <all 167,876 weights>, 'epoch': <best epoch>,
  'val_loss': <best validation MSE>}` via `torch.save`.
- **How it's reloaded**: construct `AdaptiveGRUModel` with the identical constructor
  arguments (including `residual_indices` — without it the residual anchor is silently
  disabled) and call `load_state_dict`. Step 5 above and Phase 4 both do exactly this.
- **Persistence across sessions**: `/kaggle/working` is wiped when the session ends.
  The `kagglephase3_output.zip` produced in Step 6 is what carries the checkpoints
  forward — upload it as the `kagglephase3-output` dataset so Phase 4 can load them.
